In [2]:
import pandas as pd
from pathlib import Path
import json
from collections import defaultdict

In [14]:
df = pd.read_csv("../data/processed/album_tracks.csv")

if "length_min" not in df.columns:
    df["length_min"] = df["length_ms"] / 60000

out_dir = Path("../data/processed/neo4j_import")
out_dir.mkdir(parents=True, exist_ok=True)

df.head()

,artist_id,artist_name,release_group_id,release_group_title,release_id,release_title,release_date,country,medium_format,track_position,track_title,recording_id,length_ms,length_min
0,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,1,Blew,2837e0d2-5ce9-44da-803e-797c835d672c,174133.0,2.902217
1,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,2,Floyd the Barber,4cce1ee1-38e6-4ea0-8584-b6ebeb854d7d,137973.0,2.299550
2,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,3,About a Girl,b2e623ea-378b-4479-bb75-fa202aacbfc9,168293.0,2.804883
3,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,4,School,0b3292f6-8f0d-4160-92d3-49348029a619,162173.0,2.702883
4,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,5,Love Buzz,1adfecb7-875f-4203-b3b1-8e2e643f94a2,215093.0,3.584883


In [15]:
artists = (
    df[["artist_id", "artist_name"]]
    .drop_duplicates()
    .rename(columns={"artist_name": "name"})
)

release_groups = (
    df[["release_group_id", "release_group_title"]]
    .drop_duplicates()
    .rename(columns={"release_group_title": "title"})
)

releases = (
    df[["release_id", "release_title", "release_date", "country", "medium_format"]]
    .drop_duplicates()
    .rename(columns={"release_title": "title"})
)

recordings = (
    df[["recording_id", "track_title", "length_ms", "length_min"]]
    .drop_duplicates()
    .rename(columns={"track_title": "title"})
)

In [16]:
created = (
    df[["artist_id", "release_group_id"]]
    .drop_duplicates()
)

has_release = (
    df[["release_group_id", "release_id"]]
    .drop_duplicates()
)

has_track = (
    df[["release_id", "recording_id", "track_position"]]
    .drop_duplicates()
    .rename(columns={"track_position": "position"})
)

In [22]:
artists.to_csv(out_dir / "artists.csv", index=False)
release_groups.to_csv(out_dir / "release_groups.csv", index=False)
releases.to_csv(out_dir / "releases.csv", index=False)
recordings.to_csv(out_dir / "recordings.csv", index=False)

created.to_csv(out_dir / "created.csv", index=False)
has_release.to_csv(out_dir / "has_release.csv", index=False)
has_track.to_csv(out_dir / "has_track.csv", index=False)

In [17]:
for file in out_dir.glob("*.csv"):
    temp = pd.read_csv(file)
    print(file.name, temp.shape)

artists.csv (6, 2)
created.csv (6, 2)
has_release.csv (6, 2)
has_track.csv (70, 3)
recordings.csv (70, 4)
releases.csv (6, 5)
release_groups.csv (6, 2)


In [18]:
raw_dir = Path("../data/raw/artists")
artists_files = list(raw_dir.glob("*.json"))

len(artists_files), artists_files[:2]

(6,
 [WindowsPath('../data/raw/artists/153c9281-268f-4cf3-8938-f5a4593e5df4.json'),
  WindowsPath('../data/raw/artists/5b11f4ce-a62d-471e-81fc-a69a8278c7da.json')])

In [19]:
person_rows = []
member_rows = []

member_grouped = defaultdict(lambda: {
  "person_id": None,
  "person_name": None,
  "artist_id": None,
  "artist_name": None,
  "begin": None,
  "end": None,
  "roles": set()
})

for file in artists_files:
  with open(file, "r", encoding="utf-8") as f:
    artist_data = json.load(f)

  band_id = artist_data.get("id")
  band_name = artist_data.get("name")

  for rel in artist_data.get("relations", []):
    if rel.get("type") == "member of band":
      person = rel.get("artist", {})
      
      person_id = person.get("id")
      person_name = person.get("name")

      key = (person_id, band_id, rel.get("begin"), rel.get("end"))

      member_grouped[key]["person_id"] = person_id
      member_grouped[key]["person_name"] = person_name
      member_grouped[key]["artist_id"] = band_id
      member_grouped[key]["artist_name"] = band_name
      member_grouped[key]["begin"] = rel.get("begin")
      member_grouped[key]["end"] = rel.get("end")
      member_grouped[key]["roles"].update(rel.get("attributes", []))

for data in member_grouped.values():
    person_rows.append({
      "person_id": data["person_id"],
      "name": data["person_name"]
    })

    member_rows.append({
      "person_id": data["person_id"],
      "artist_id": data["artist_id"],
      "begin": data["begin"],
      "end": data["end"],
      "roles": ";".join(sorted(data["roles"]))
    })

persons = pd.DataFrame(person_rows).drop_duplicates()
member_of = pd.DataFrame(member_rows).drop_duplicates()

persons.head(), member_of.head()

(                              person_id           name
 0  fb7687e9-067d-4b5b-ac66-0d7a184c6034  Hiro Yamamoto
 1  cbf9738d-8f81-4a92-bc64-ede09341652d  Chris Cornell
 2  dd676aae-658e-42f3-b3d9-8660ea29198d     Kim Thayil
 3  07e93890-46a1-4ebd-9a70-3988a2135d87   Matt Cameron
 4  797dc323-fc39-4597-98cc-d10493f67326  Jason Everman,
                               person_id                             artist_id  \
 0  fb7687e9-067d-4b5b-ac66-0d7a184c6034  153c9281-268f-4cf3-8938-f5a4593e5df4   
 1  cbf9738d-8f81-4a92-bc64-ede09341652d  153c9281-268f-4cf3-8938-f5a4593e5df4   
 2  dd676aae-658e-42f3-b3d9-8660ea29198d  153c9281-268f-4cf3-8938-f5a4593e5df4   
 3  07e93890-46a1-4ebd-9a70-3988a2135d87  153c9281-268f-4cf3-8938-f5a4593e5df4   
 4  797dc323-fc39-4597-98cc-d10493f67326  153c9281-268f-4cf3-8938-f5a4593e5df4   
 
   begin         end                          roles  
 0  1984        1989  electric bass guitar;original  
 1  1984  2017-05-18    guitar;lead vocals;original  
 2  198

In [20]:
tag_rows = []
has_tag_rows = []

for file in artists_files:
  with open(file, "r", encoding="utf-8") as f:
    artist_data = json.load(f)

  artist_id = artist_data.get("id")

  for tag in artist_data.get("tags", []):
    tag_name = tag.get("name")
    tag_count = tag.get("count")

    if tag_name:
      tag_rows.append({
        "name": tag_name
      })

    has_tag_rows.append({
      "artist_id": artist_id,
      "tag": tag_name,
      "count": tag_count
    })

tags = pd.DataFrame(tag_rows).drop_duplicates()
has_tag = pd.DataFrame(has_tag_rows).drop_duplicates()

tags.head(), has_tag.head()

(                name
 0  alternative metal
 1   alternative rock
 2           american
 3             grunge
 4          hard rock,
                               artist_id                tag  count
 0  153c9281-268f-4cf3-8938-f5a4593e5df4  alternative metal     14
 1  153c9281-268f-4cf3-8938-f5a4593e5df4   alternative rock     18
 2  153c9281-268f-4cf3-8938-f5a4593e5df4           american      2
 3  153c9281-268f-4cf3-8938-f5a4593e5df4             grunge     20
 4  153c9281-268f-4cf3-8938-f5a4593e5df4          hard rock      9)

In [23]:
persons.to_csv(out_dir / "persons.csv", index=False)
member_of.to_csv(out_dir / "member_of.csv", index=False)
tags.to_csv(out_dir / "tags.csv", index=False)
has_tag.to_csv(out_dir / "has_tag.csv", index=False)

In [24]:
for file in sorted(out_dir.glob("*.csv")):
    temp = pd.read_csv(file)
    print(file.name, temp.shape)

artists.csv (6, 2)
created.csv (6, 2)
has_release.csv (6, 2)
has_tag.csv (75, 3)
has_track.csv (70, 3)
member_of.csv (52, 5)
persons.csv (44, 2)
recordings.csv (70, 4)
release_groups.csv (6, 2)
releases.csv (6, 5)
tags.csv (45, 1)


#### Sanity Check

In [25]:
artists = pd.read_csv(out_dir / "artists.csv")
release_groups = pd.read_csv(out_dir / "release_groups.csv")
releases = pd.read_csv(out_dir / "releases.csv")
recordings = pd.read_csv(out_dir / "recordings.csv")
persons = pd.read_csv(out_dir / "persons.csv")
tags = pd.read_csv(out_dir / "tags.csv")

created = pd.read_csv(out_dir / "created.csv")
has_release = pd.read_csv(out_dir / "has_release.csv")
has_track = pd.read_csv(out_dir / "has_track.csv")
member_of = pd.read_csv(out_dir / "member_of.csv")
has_tag = pd.read_csv(out_dir / "has_tag.csv")

In [26]:
csvs = {
    "artists": artists,
    "release_groups": release_groups,
    "releases": releases,
    "recordings": recordings,
    "persons": persons,
    "tags": tags,
    "created": created,
    "has_release": has_release,
    "has_track": has_track,
    "member_of": member_of,
    "has_tag": has_tag,
}

for name, table in csvs.items():
    print(f"\n{name}")
    print(table.isna().sum())


artists
artist_id    0
name         0
dtype: int64

release_groups
release_group_id    0
title               0
dtype: int64

releases
release_id       0
title            0
release_date     0
country          0
medium_format    0
dtype: int64

recordings
recording_id    0
title           0
length_ms       1
length_min      1
dtype: int64

persons
person_id    0
name         0
dtype: int64

tags
name    0
dtype: int64

created
artist_id           0
release_group_id    0
dtype: int64

has_release
release_group_id    0
release_id          0
dtype: int64

has_track
release_id      0
recording_id    0
position        0
dtype: int64

member_of
person_id     0
artist_id     0
begin         4
end          22
roles         1
dtype: int64

has_tag
artist_id    0
tag          0
count        0
dtype: int64


In [27]:
checks = {
    "created.artist_id missing from artists": set(created["artist_id"]) - set(artists["artist_id"]),
    "created.release_group_id missing from release_groups": set(created["release_group_id"]) - set(release_groups["release_group_id"]),

    "has_release.release_group_id missing from release_groups": set(has_release["release_group_id"]) - set(release_groups["release_group_id"]),
    "has_release.release_id missing from releases": set(has_release["release_id"]) - set(releases["release_id"]),

    "has_track.release_id missing from releases": set(has_track["release_id"]) - set(releases["release_id"]),
    "has_track.recording_id missing from recordings": set(has_track["recording_id"]) - set(recordings["recording_id"]),

    "member_of.person_id missing from persons": set(member_of["person_id"]) - set(persons["person_id"]),
    "member_of.artist_id missing from artists": set(member_of["artist_id"]) - set(artists["artist_id"]),

    "has_tag.artist_id missing from artists": set(has_tag["artist_id"]) - set(artists["artist_id"]),
    "has_tag.tag missing from tags": set(has_tag["tag"]) - set(tags["name"]),
}

for check_name, missing_values in checks.items():
    print(check_name, ":", len(missing_values))

created.artist_id missing from artists : 0
created.release_group_id missing from release_groups : 0
has_release.release_group_id missing from release_groups : 0
has_release.release_id missing from releases : 0
has_track.release_id missing from releases : 0
has_track.recording_id missing from recordings : 0
member_of.person_id missing from persons : 0
member_of.artist_id missing from artists : 0
has_tag.artist_id missing from artists : 0
has_tag.tag missing from tags : 0


In [28]:
print("duplicate artist_id:", artists["artist_id"].duplicated().sum())
print("duplicate release_group_id:", release_groups["release_group_id"].duplicated().sum())
print("duplicate release_id:", releases["release_id"].duplicated().sum())
print("duplicate recording_id:", recordings["recording_id"].duplicated().sum())
print("duplicate person_id:", persons["person_id"].duplicated().sum())
print("duplicate tag name:", tags["name"].duplicated().sum())

duplicate artist_id: 0
duplicate release_group_id: 0
duplicate release_id: 0
duplicate recording_id: 0
duplicate person_id: 0
duplicate tag name: 0


#### Neo4j Import CSV Sanity Check Conclusion

All generated relationship tables reference valid node IDs. No dangling relationships were found.

All node identifier columns are unique:
- artist_id
- release_group_id
- release_id
- recording_id
- person_id
- tag name

`recordings.csv` still contains 1 missing `length_ms` / `length_min` value from the original MusicBrainz metadata. This is kept as missing to preserve source consistency.